# 🍛 NutriVision — YOLOv8 Kerala / South Indian Food Detection
## Complete Training Notebook for Kaggle GPU

### Target food classes (20 classes):
| Class | Description |
|---|---|
| `rice` | Plain cooked rice (white/red) |
| `sambar` | Sambar (bowl or side) |
| `aviyal` | Aviyal (mixed veg Kerala curry) |
| `idli` | Idli (steamed rice cakes) |
| `coconut_chutney` | White coconut chutney |
| `green_chutney` | Green/mint chutney |
| `chicken_curry` | Kerala chicken curry |
| `biryani` | Biryani (chicken/mutton/veg) |
| `chapathi` | Chapathi / roti |
| `dal_curry` | Dal / lentil curry |
| `kadala_curry` | Black chickpea curry |
| `appam` | Appam (Kerala rice pancake) |
| `idiyappam` | Idiyappam (string hoppers) |
| `porotta` | Kerala parotta (layered flatbread) |
| `beef_curry` | Kerala beef curry |
| `fish_curry` | Kerala fish curry |
| `egg_curry` | Egg curry |
| `dosa` | Plain / masala dosa |
| `vada` | Medu vada / parippu vada |
| `puttu` | Puttu with kadala curry |

---
**Kaggle Settings needed:**
- Accelerator: **GPU P100** (free tier) or T4 x2
- Internet: **ON** (for Roboflow download)
- RAM: 13GB+ (P100 gives 16GB VRAM)

**Runtime:** ~3–5 hours for 100 epochs on P100

---
## CELL 1 — Check GPU and environment

In [ ]:
import subprocess
import os

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Check available disk
disk = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(disk.stdout)

# Check RAM
mem = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(mem.stdout)

print("\n✅ Environment check done")

---
## CELL 2 — Install dependencies

In [ ]:
# Install Ultralytics YOLOv8 and Roboflow
!pip install -q ultralytics roboflow opencv-python-headless pillow

# Verify ultralytics version
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

from ultralytics import YOLO
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n✅ All packages installed")

---
## CELL 3 — Set up working directories

In [ ]:
import os
import shutil
from pathlib import Path

# Base directory for all data
BASE_DIR = Path('/kaggle/working/nutrivision')
DATASET_DIR = BASE_DIR / 'datasets'
COMBINED_DIR = BASE_DIR / 'combined_dataset'
RUNS_DIR = BASE_DIR / 'runs'

# Create directory structure
for split in ['train', 'val', 'test']:
    (COMBINED_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (COMBINED_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

DATASET_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {BASE_DIR}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Combined dataset: {COMBINED_DIR}")
print("\n✅ Directories created")

---
## CELL 4 — CLASS DEFINITIONS

These are your 20 target food classes.
**CRITICAL:** The order here determines class IDs (0–19) in all label files.

In [ ]:
# ━━━ YOUR 20 TARGET CLASSES ━━━
# These match exactly what NutriVision backend expects
CLASS_NAMES = [
    'rice',           # 0  — plain cooked rice
    'sambar',         # 1  — sambar
    'aviyal',         # 2  — aviyal
    'idli',           # 3  — idli
    'coconut_chutney',# 4  — white coconut chutney
    'green_chutney',  # 5  — green/mint chutney
    'chicken_curry',  # 6  — Kerala chicken curry
    'biryani',        # 7  — biryani
    'chapathi',       # 8  — chapathi/roti
    'dal_curry',      # 9  — dal/lentil curry
    'kadala_curry',   # 10 — black chickpea curry
    'appam',          # 11 — appam
    'idiyappam',      # 12 — idiyappam
    'porotta',        # 13 — Kerala parotta
    'beef_curry',     # 14 — Kerala beef curry
    'fish_curry',     # 15 — Kerala fish curry
    'egg_curry',      # 16 — egg curry
    'dosa',           # 17 — dosa
    'vada',           # 18 — medu vada / parippu vada
    'puttu',          # 19 — puttu
]

NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print(f"Total classes: {NUM_CLASSES}")
print("\nClass ID mapping:")
for name, idx in CLASS_TO_ID.items():
    print(f"  {idx:2d}: {name}")

---
## CELL 5 — Download datasets from Roboflow

### Dataset Strategy (why these specific datasets):

| Dataset | Roboflow Link | Key Classes Covered | Images |
|---|---|---|---|
| **South Indian Food Detection** (Bharani) | `bharani-1ucid/south-indian-food-detection` | appam, idly, dosa, sambar, chutney, biryani, vada | ~500 |
| **Food Recognition** (Credo Health) | `credo-health/food-recognition-ow7id` | idiyappam, idli, sambar, rice, dosa, chutney | ~4895 |
| **Indian Food Detection** (project) | `project-z0tql/indian-food-detection-kzw9g` | chicken curry, biryani, chapathi, dal, egg curry | ~4133 |
| **South Indian Food (SIFD)** | `south-indian-food-detection-and-classification/food-detection-nlusn` | appam, sambar, chutney, rice, dosa | ~600 |

**Note:** appam, idiyappam, porotta, aviyal, kadala curry, beef curry are **rare classes**. The notebook handles this with aggressive augmentation and class-weighted loss.

In [ ]:
from roboflow import Roboflow

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1: Get your free Roboflow API key:
#   1. Go to https://roboflow.com
#   2. Sign up (free)
#   3. Go to Settings → API Key
#   4. Paste it below
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print("✅ Roboflow authenticated")

In [ ]:
import json
import shutil

downloaded_datasets = []

# ━━━ DATASET 1: South Indian Food Detection (Bharani) ━━━
# Has: appam, dosa, idly, sambar, chutney, biryani, vada — VERY relevant
print("Downloading Dataset 1: South Indian Food Detection...")
try:
    project = rf.workspace("bharani-1ucid").project("south-indian-food-detection")
    dataset = project.version(1).download("yolov8", location=str(DATASET_DIR / "ds1"))
    downloaded_datasets.append((DATASET_DIR / "ds1", "south_indian"))
    print("  ✅ Dataset 1 downloaded")
except Exception as e:
    print(f"  ⚠️  Dataset 1 failed: {e}")
    print("  Trying alternative URL...")
    try:
        project = rf.workspace("south-indian-food-detection-and-classification").project("food-detection-nlusn")
        dataset = project.version(1).download("yolov8", location=str(DATASET_DIR / "ds1"))
        downloaded_datasets.append((DATASET_DIR / "ds1", "south_indian_alt"))
        print("  ✅ Dataset 1 (alt) downloaded")
    except Exception as e2:
        print(f"  ❌ Both failed: {e2}")

print()

In [ ]:
# ━━━ DATASET 2: Food Recognition by Credo Health ━━━
# Has: idiyappam, idli, sambar, rice, dosa, chutney — CRITICAL for idiyappam
print("Downloading Dataset 2: Food Recognition (Credo Health)...")
try:
    project = rf.workspace("credo-health").project("food-recognition-ow7id")
    dataset = project.version(1).download("yolov8", location=str(DATASET_DIR / "ds2"))
    downloaded_datasets.append((DATASET_DIR / "ds2", "credo_health"))
    print("  ✅ Dataset 2 downloaded")
except Exception as e:
    print(f"  ⚠️  Dataset 2 failed: {e}")

print()

In [ ]:
# ━━━ DATASET 3: Indian Food Detection (project-z0tql) ━━━
# 4133 images — has chicken curry, biryani, chapathi, dal, egg curry
print("Downloading Dataset 3: Indian Food Detection (4133 images)...")
try:
    project = rf.workspace("project-z0tql").project("indian-food-detection-kzw9g")
    dataset = project.version(1).download("yolov8", location=str(DATASET_DIR / "ds3"))
    downloaded_datasets.append((DATASET_DIR / "ds3", "indian_food"))
    print("  ✅ Dataset 3 downloaded")
except Exception as e:
    print(f"  ⚠️  Dataset 3 failed: {e}")

print()

In [ ]:
# ━━━ DATASET 4: Indian Food Recognition and Labelling (college) ━━━
# 1849 images — broad Indian food coverage
print("Downloading Dataset 4: Indian Food Recognition & Labelling...")
try:
    project = rf.workspace("college-fdisg").project("indian-food-recognition-and-labelling")
    dataset = project.version(1).download("yolov8", location=str(DATASET_DIR / "ds4"))
    downloaded_datasets.append((DATASET_DIR / "ds4", "food_labelling"))
    print("  ✅ Dataset 4 downloaded")
except Exception as e:
    print(f"  ⚠️  Dataset 4 failed: {e}")

print(f"\n✅ Downloaded {len(downloaded_datasets)} datasets")
for path, name in downloaded_datasets:
    print(f"  - {name}: {path}")

---
## CELL 6 — Class Mapping

Each Roboflow dataset uses its own class names and IDs.
This cell maps THEIR class names → YOUR 20 class IDs.
Classes not in your target list are discarded (label files become empty for those images).

In [ ]:
# ━━━ MASTER CLASS MAPPING ━━━
# Maps source dataset class names → your NutriVision class names
# All variations, spellings, and aliases are handled here

SOURCE_TO_TARGET = {
    # ── RICE ──
    'rice': 'rice', 'Rice': 'rice', 'satham': 'rice', 'Satham': 'rice',
    'white rice': 'rice', 'cooked rice': 'rice', 'plain rice': 'rice',
    'nei satham': 'rice', 'lemon satham': 'rice',

    # ── SAMBAR ──
    'sambar': 'sambar', 'Sambar': 'sambar', 'sambhar': 'sambar',
    'sambar satham': 'sambar',

    # ── AVIYAL ──
    'aviyal': 'aviyal', 'Aviyal': 'aviyal', 'avial': 'aviyal',

    # ── IDLI ──
    'idli': 'idli', 'Idli': 'idli', 'idly': 'idli', 'Idly': 'idli',
    'fried idli': 'idli', 'rava idli': 'idli', 'samai idli': 'idli',

    # ── COCONUT CHUTNEY ──
    'coconut chutney': 'coconut_chutney', 'Coconut Chutney': 'coconut_chutney',
    'thengai chutney': 'coconut_chutney', 'Thengai chutney': 'coconut_chutney',
    'white chutney': 'coconut_chutney',
    'chutney': 'coconut_chutney',  # default chutney → coconut

    # ── GREEN CHUTNEY ──
    'green chutney': 'green_chutney', 'Green Chutney': 'green_chutney',
    'puthina chutney': 'green_chutney', 'Puthina Chutney': 'green_chutney',
    'mint chutney': 'green_chutney', 'kaara chutney': 'green_chutney',
    'Kaara chutney': 'green_chutney', 'Kaara Chutney': 'green_chutney',

    # ── CHICKEN CURRY ──
    'chicken curry': 'chicken_curry', 'Chicken Curry': 'chicken_curry',
    'chicken briyani': 'biryani',  # biryani handled separately
    'chicken 65': 'chicken_curry', 'Chicken 65': 'chicken_curry',
    'butter chicken': 'chicken_curry',

    # ── BIRYANI ──
    'biryani': 'biryani', 'Biryani': 'biryani', 'biriyani': 'biryani',
    'Biriyani': 'biryani', 'briyani': 'biryani',
    'chicken briyani': 'biryani', 'Chicken briyani': 'biryani',
    'mutton briyani': 'biryani', 'Mutton Briyani': 'biryani',
    'veg briyani': 'biryani', 'Veg briyani': 'biryani',
    'paneer briyani': 'biryani', 'Paneer briyani': 'biryani',
    'mushroom briyani': 'biryani',
    'chicken_biryani': 'biryani', 'BIRIYANI': 'biryani',

    # ── CHAPATHI ──
    'chapathi': 'chapathi', 'Chapathi': 'chapathi', 'chapati': 'chapathi',
    'CHAPATHI': 'chapathi', 'roti': 'chapathi', 'Roti': 'chapathi',
    'phulka': 'chapathi',

    # ── DAL CURRY ──
    'dal': 'dal_curry', 'Dal': 'dal_curry', 'dal curry': 'dal_curry',
    'DAL': 'dal_curry', 'lentil curry': 'dal_curry',
    'parupu vadai': 'vada',  # redirect to vada class

    # ── KADALA CURRY ──
    'kadala curry': 'kadala_curry', 'Kadala Curry': 'kadala_curry',
    'kadala': 'kadala_curry', 'black chana curry': 'kadala_curry',
    'chana masala': 'kadala_curry', 'chole': 'kadala_curry',

    # ── APPAM ──
    'appam': 'appam', 'Appam': 'appam', 'APPAM': 'appam',
    'aapam': 'appam', 'AAPAM': 'appam',

    # ── IDIYAPPAM ──
    'idiyappam': 'idiyappam', 'Idiyappam': 'idiyappam', 'IDIYAPPAM': 'idiyappam',
    'string hoppers': 'idiyappam', 'neer idiyappam': 'idiyappam',

    # ── POROTTA ──
    'porotta': 'porotta', 'Porotta': 'porotta', 'parotta': 'porotta',
    'Parotta': 'porotta', 'paratha': 'porotta', 'Paratha': 'porotta',
    'POROTTA': 'porotta', 'kerala parotta': 'porotta',

    # ── BEEF CURRY ──
    'beef curry': 'beef_curry', 'Beef Curry': 'beef_curry',
    'beef': 'beef_curry', 'BEEF CURRY': 'beef_curry',
    'beef fry': 'beef_curry', 'nadan beef': 'beef_curry',

    # ── FISH CURRY ──
    'fish curry': 'fish_curry', 'Fish Curry': 'fish_curry',
    'FISH CURRY': 'fish_curry', 'fish fry': 'fish_curry',
    'kerala fish curry': 'fish_curry', 'FISH FRY': 'fish_curry',

    # ── EGG CURRY ──
    'egg curry': 'egg_curry', 'Egg Curry': 'egg_curry',
    'egg kuzhambu': 'egg_curry', 'Egg Kuzhambu': 'egg_curry',
    'boiled egg': 'egg_curry', 'Boiled Egg': 'egg_curry', 'boiled egg': 'egg_curry',

    # ── DOSA ──
    'dosa': 'dosa', 'Dosa': 'dosa', 'DOSA': 'dosa',
    'masala dosa': 'dosa', 'plain dosa': 'dosa', 'neer dosa': 'dosa',
    'rava dosa': 'dosa', 'mixed veg dosa': 'dosa',

    # ── VADA ──
    'vada': 'vada', 'Vada': 'vada', 'medu vadai': 'vada',
    'medu vada': 'vada', 'Medu vadai': 'vada', 'Medu Vada': 'vada',
    'uzhuntha vadai': 'vada', 'parupu vadai': 'vada',

    # ── PUTTU ──
    'puttu': 'puttu', 'Puttu': 'puttu', 'PUTTU': 'puttu',
    'rice puttu': 'puttu', 'wheat puttu': 'puttu',
}

print(f"Class mappings defined: {len(SOURCE_TO_TARGET)} source names")
print(f"→ Target classes: {NUM_CLASSES}")

---
## CELL 7 — Merge and remap all datasets

In [ ]:
import yaml
import shutil
import glob
from pathlib import Path
from collections import defaultdict

def read_dataset_yaml(dataset_path):
    """Read the data.yaml from a Roboflow download and return class names."""
    yaml_files = list(Path(dataset_path).glob('*.yaml')) + list(Path(dataset_path).glob('data.yaml'))
    if not yaml_files:
        yaml_files = list(Path(dataset_path).rglob('data.yaml'))
    if not yaml_files:
        print(f"  ⚠️  No YAML found in {dataset_path}")
        return {}
    
    with open(yaml_files[0], 'r') as f:
        data = yaml.safe_load(f)
    
    # Build source class id → source class name mapping
    names = data.get('names', {})
    if isinstance(names, list):
        return {i: name for i, name in enumerate(names)}
    return names  # already a dict


def remap_label_file(src_label_path, src_id_to_name, target_id_mapping):
    """
    Read a YOLO label file, remap class IDs using source→target mapping.
    Returns list of remapped lines (empty list if no valid classes).
    """
    new_lines = []
    try:
        with open(src_label_path, 'r') as f:
            lines = f.readlines()
    except:
        return []
    
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        
        src_class_id = int(parts[0])
        src_class_name = src_id_to_name.get(src_class_id, '')
        
        # Try direct mapping, then lowercase
        target_name = SOURCE_TO_TARGET.get(src_class_name) or \
                      SOURCE_TO_TARGET.get(src_class_name.lower())
        
        if target_name is None:
            continue  # Discard classes not in our target list
        
        target_id = CLASS_TO_ID[target_name]
        new_line = f"{target_id} {' '.join(parts[1:])}"
        new_lines.append(new_line)
    
    return new_lines


def copy_and_remap_dataset(dataset_path, split, src_id_to_name, file_prefix):
    """
    Copy images + remapped labels from one dataset split into the combined dataset.
    Returns count of (images_copied, labels_with_annotations)
    """
    dataset_path = Path(dataset_path)
    img_src = dataset_path / split / 'images'
    lbl_src = dataset_path / split / 'labels'
    
    if not img_src.exists():
        # Try alternate structure
        img_src = dataset_path / 'images' / split
        lbl_src = dataset_path / 'labels' / split
    
    if not img_src.exists():
        return 0, 0
    
    img_dst = COMBINED_DIR / 'images' / split
    lbl_dst = COMBINED_DIR / 'labels' / split
    
    images_copied = 0
    annotated = 0
    
    image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
    
    for img_file in img_src.iterdir():
        if img_file.suffix not in image_extensions:
            continue
        
        # Find corresponding label file
        lbl_file = lbl_src / (img_file.stem + '.txt')
        
        # Remap labels
        new_lines = []
        if lbl_file.exists():
            new_lines = remap_label_file(lbl_file, src_id_to_name, CLASS_TO_ID)
        
        # Copy image with unique prefix to avoid name conflicts
        new_img_name = f"{file_prefix}_{img_file.name}"
        shutil.copy2(img_file, img_dst / new_img_name)
        images_copied += 1
        
        # Write remapped label
        new_lbl_name = f"{file_prefix}_{img_file.stem}.txt"
        with open(lbl_dst / new_lbl_name, 'w') as f:
            f.write('\n'.join(new_lines))
        
        if new_lines:
            annotated += 1
    
    return images_copied, annotated


# ━━━ MERGE ALL DOWNLOADED DATASETS ━━━
total_stats = defaultdict(lambda: defaultdict(int))

for ds_idx, (ds_path, ds_name) in enumerate(downloaded_datasets):
    print(f"\nProcessing dataset: {ds_name}")
    src_classes = read_dataset_yaml(ds_path)
    print(f"  Source classes ({len(src_classes)}): {list(src_classes.values())[:10]}...")
    
    prefix = f"ds{ds_idx+1}"
    for split in ['train', 'valid', 'test']:
        target_split = 'val' if split == 'valid' else split
        imgs, annotated = copy_and_remap_dataset(ds_path, split, src_classes, f"{prefix}_{split}")
        if imgs > 0:
            print(f"  [{split}] → Copied: {imgs} images, Annotated: {annotated}")
            total_stats['total'][target_split] += imgs
            total_stats['annotated'][target_split] += annotated

print("\n" + "="*50)
print("COMBINED DATASET SUMMARY:")
for split in ['train', 'val', 'test']:
    total = total_stats['total'][split]
    ann = total_stats['annotated'][split]
    print(f"  {split:6s}: {total:5d} images, {ann:5d} annotated ({ann/max(total,1)*100:.1f}%)")

In [ ]:
# ━━━ Count class distribution in final combined dataset ━━━
from collections import Counter

def count_class_distribution(split):
    lbl_dir = COMBINED_DIR / 'labels' / split
    counter = Counter()
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    return counter

print("Class distribution in TRAIN split:")
dist = count_class_distribution('train')
print(f"{'Class ID':>10} {'Class Name':>20} {'Count':>8}")
print("-" * 42)
for cls_id, name in enumerate(CLASS_NAMES):
    count = dist.get(cls_id, 0)
    bar = '█' * min(count // 5, 30)
    flag = " ⚠️  RARE" if count < 50 else ""
    print(f"{cls_id:>10} {name:>20} {count:>8}  {bar}{flag}")

print(f"\nTotal annotations in train: {sum(dist.values())}")

rare_classes = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if dist.get(i, 0) < 50]
if rare_classes:
    print(f"\n⚠️  Rare classes needing manual images: {rare_classes}")

---
## CELL 8 — Scrape additional images for rare classes

For classes like **aviyal, kadala_curry, porotta, idiyappam, puttu, beef_curry** that have very few images, we use `icrawler` to download more from Google Images, then you manually annotate them using Label Studio or Roboflow.

In [ ]:
!pip install -q icrawler

from icrawler.builtin import GoogleImageCrawler, BingImageCrawler
import os

SCRAPE_DIR = BASE_DIR / 'scraped_images'

# Search queries for each rare class — specific for Kerala/South Indian food
SCRAPE_QUERIES = {
    'aviyal':       ['Kerala aviyal vegetable curry', 'aviyal Kerala sadya', 'aviyal mixed vegetables'],
    'kadala_curry': ['Kerala kadala curry', 'black chickpea curry Kerala', 'kadala puttu'],
    'porotta':      ['Kerala parotta', 'malabar porotta layered', 'Kerala porotta beef'],
    'idiyappam':    ['Kerala idiyappam', 'string hoppers South India', 'idiyappam with curry'],
    'puttu':        ['Kerala puttu', 'puttu kadala Kerala', 'puttu banana',
                     'rice puttu steamed cylindrical'],
    'beef_curry':   ['Kerala beef curry', 'nadan beef fry Kerala', 'Kerala beef roast'],
    'appam':        ['Kerala appam', 'appam with stew', 'Kerala appam bowl shaped'],
}

IMAGES_PER_QUERY = 60  # per search query

print("Scraping images for rare classes...")
print("NOTE: These images MUST be manually annotated in Roboflow/Label Studio before use!")
print()

for class_name, queries in SCRAPE_QUERIES.items():
    class_dir = SCRAPE_DIR / class_name
    class_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Scraping: {class_name}")
    for query in queries:
        try:
            crawler = GoogleImageCrawler(
                storage={'root_dir': str(class_dir)},
                feeder_threads=1,
                parser_threads=2,
                downloader_threads=4,
            )
            crawler.crawl(
                keyword=query,
                max_num=IMAGES_PER_QUERY,
                min_size=(200, 200),
                file_idx_offset='auto',
            )
        except Exception as e:
            print(f"  ⚠️  Failed for '{query}': {e}")
    
    count = len(list(class_dir.glob('*.jpg'))) + len(list(class_dir.glob('*.png')))
    print(f"  → Downloaded ~{count} images to {class_dir}")

print("\n✅ Scraping done")
print("\n📋 NEXT STEP:")
print("   Go to https://roboflow.com → New Project → Upload scraped images → Annotate → Export YOLOv8")
print(f"   Scraped images are in: {SCRAPE_DIR}")

---
## CELL 9 — Write the dataset YAML config

In [ ]:
import yaml

# Count actual images to confirm paths
def count_images(split):
    img_dir = COMBINED_DIR / 'images' / split
    if not img_dir.exists():
        return 0
    return len([f for f in img_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])

train_count = count_images('train')
val_count = count_images('val')
test_count = count_images('test')

print(f"Images — train: {train_count}, val: {val_count}, test: {test_count}")

# Fallback: if val is empty, split 15% from train
if val_count == 0 and train_count > 0:
    print("\nNo validation split found — creating 15% val from train...")
    import random
    train_imgs = list((COMBINED_DIR / 'images' / 'train').glob('*'))
    val_size = max(1, int(len(train_imgs) * 0.15))
    val_imgs = random.sample(train_imgs, val_size)
    
    for img_path in val_imgs:
        # Move image
        dst_img = COMBINED_DIR / 'images' / 'val' / img_path.name
        shutil.move(str(img_path), str(dst_img))
        # Move label
        lbl_path = COMBINED_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
        dst_lbl = COMBINED_DIR / 'labels' / 'val' / (img_path.stem + '.txt')
        if lbl_path.exists():
            shutil.move(str(lbl_path), str(dst_lbl))
    
    val_count = count_images('val')
    train_count = count_images('train')
    print(f"  New split — train: {train_count}, val: {val_count}")

# Write data.yaml
data_yaml = {
    'path': str(COMBINED_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES,
}

YAML_PATH = COMBINED_DIR / 'data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ data.yaml written to {YAML_PATH}")
print("\nContents:")
with open(YAML_PATH) as f:
    print(f.read())

---
## CELL 10 — Data augmentation for rare classes

For classes with fewer than 80 annotated examples, we apply offline augmentation to boost them before training.

In [ ]:
import cv2
import numpy as np
import random
from pathlib import Path

def augment_image(img: np.ndarray):
    """Apply random augmentations: flip, rotate, brightness, hue shift."""
    augmented = []
    
    # 1. Horizontal flip
    augmented.append(cv2.flip(img, 1))
    
    # 2. Brightness / contrast
    alpha = random.uniform(0.7, 1.3)  # contrast
    beta = random.randint(-30, 30)     # brightness
    augmented.append(np.clip(alpha * img + beta, 0, 255).astype(np.uint8))
    
    # 3. HSV shift (hue jitter)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:,:,0] = (hsv[:,:,0] + random.randint(-15, 15)) % 180
    hsv[:,:,1] = np.clip(hsv[:,:,1] * random.uniform(0.8, 1.2), 0, 255)
    augmented.append(cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR))
    
    # 4. Slight rotation (±10°)
    h, w = img.shape[:2]
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    augmented.append(cv2.warpAffine(img, M, (w, h)))
    
    return augmented


def flip_bbox_horizontal(bbox_line):
    """Flip a YOLO bbox horizontally: cx = 1 - cx, rest unchanged."""
    parts = bbox_line.strip().split()
    if len(parts) < 5:
        return bbox_line
    cls_id = parts[0]
    cx = 1.0 - float(parts[1])
    cy, w, h = float(parts[2]), float(parts[3]), float(parts[4])
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def augment_rare_class_images(class_id, target_count=150):
    """Augment images of a rare class until we reach target_count instances."""
    lbl_dir = COMBINED_DIR / 'labels' / 'train'
    img_dir = COMBINED_DIR / 'images' / 'train'
    
    # Find all label files containing this class
    candidate_files = []
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file) as f:
            content = f.read()
        if any(line.split()[0] == str(class_id) for line in content.splitlines() if line.strip()):
            candidate_files.append(lbl_file)
    
    current_count = len(candidate_files)
    if current_count == 0:
        return 0
    if current_count >= target_count:
        return current_count
    
    needed = target_count - current_count
    aug_count = 0
    
    while aug_count < needed:
        lbl_file = random.choice(candidate_files)
        
        # Find corresponding image
        img_file = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            candidate = img_dir / (lbl_file.stem + ext)
            if candidate.exists():
                img_file = candidate
                break
        
        if img_file is None:
            continue
        
        img = cv2.imread(str(img_file))
        if img is None:
            continue
        
        with open(lbl_file) as f:
            label_lines = f.readlines()
        
        augmented_imgs = augment_image(img)
        
        for aug_idx, aug_img in enumerate(augmented_imgs):
            if aug_count >= needed:
                break
            
            new_stem = f"aug_{class_id}_{aug_count:05d}_{lbl_file.stem}"
            
            # Save augmented image
            cv2.imwrite(str(img_dir / (new_stem + '.jpg')), aug_img)
            
            # Save (flipped if aug_idx==0) labels
            if aug_idx == 0:  # horizontal flip
                new_lines = [flip_bbox_horizontal(l) for l in label_lines if l.strip()]
            else:
                new_lines = [l.strip() for l in label_lines if l.strip()]
            
            with open(lbl_dir / (new_stem + '.txt'), 'w') as f:
                f.write('\n'.join(new_lines))
            
            aug_count += 1
    
    return current_count + aug_count


# Run augmentation for rare classes
dist = count_class_distribution('train')
print("Augmenting rare classes (target: 150 instances each)...\n")

for cls_id, cls_name in enumerate(CLASS_NAMES):
    count = dist.get(cls_id, 0)
    if count < 150:
        new_count = augment_rare_class_images(cls_id, target_count=150)
        print(f"  {cls_name:25s}: {count} → {new_count} instances")
    else:
        print(f"  {cls_name:25s}: {count} (OK)")

print("\n✅ Augmentation done")

---
## CELL 11 — Training Configuration

All hyperparameters are tuned for:
- Multi-dish plate scenes (multiple small objects per image)
- Rare classes (mosaic + mixup augmentation)
- Kaggle P100 GPU (16GB VRAM)

In [ ]:
# ━━━ TRAINING CONFIGURATION ━━━
TRAIN_CONFIG = {
    # ── Model ──
    'model':     'yolov8n.pt',  # nano — fastest, smallest, great for mobile
                                # Alternatives: yolov8s.pt (better accuracy, slower)
    
    # ── Data ──
    'data':      str(YAML_PATH),
    'project':   str(RUNS_DIR),
    'name':      'nutrivision_v1',
    
    # ── Training epochs ──
    'epochs':    100,           # 100 epochs is standard; increase to 150 if val mAP still rising
    'patience':  25,            # early stopping: stop if no improvement for 25 epochs
    
    # ── Image size ──
    'imgsz':     640,           # standard YOLOv8 input size
    
    # ── Batch size ──
    # P100 (16GB): batch=32 is safe with yolov8n
    # T4 (15GB): batch=16
    'batch':     32,
    
    # ── Optimizer ──
    'optimizer': 'AdamW',       # AdamW > SGD for small datasets
    'lr0':       0.001,         # initial learning rate
    'lrf':       0.01,          # final lr = lr0 × lrf
    'momentum':  0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    
    # ── Augmentation ──
    # These are ON by default in YOLOv8; explicitly set them
    'mosaic':    1.0,           # CRITICAL for multi-dish plates — creates composite images
    'mixup':     0.1,           # mild mixup augmentation
    'copy_paste': 0.1,          # copy-paste objects across images
    'degrees':   10.0,          # ±10° rotation
    'translate': 0.1,           # ±10% translation
    'scale':     0.5,           # scale ±50%
    'shear':     2.0,           # shear ±2°
    'perspective': 0.0,         # no perspective distortion for food
    'flipud':    0.0,           # no vertical flip (food plates have orientation)
    'fliplr':    0.5,           # 50% horizontal flip
    'hsv_h':     0.015,         # hue shift ±1.5%
    'hsv_s':     0.7,           # saturation shift ±70% (food color varies a lot)
    'hsv_v':     0.4,           # value (brightness) shift ±40%
    
    # ── Hardware ──
    'device':    0,             # GPU 0
    'workers':   4,
    'amp':       True,          # automatic mixed precision — saves VRAM
    
    # ── Checkpointing ──
    'save':      True,
    'save_period': 10,          # save checkpoint every 10 epochs
    
    # ── Validation ──
    'val':       True,
    'plots':     True,          # save training plots
    
    # ── Thresholds ──
    'conf':      0.35,          # minimum confidence for detection
    'iou':       0.5,           # NMS IoU threshold
    'max_det':   20,            # max detections per image (one plate = max 10 items)
}

print("Training Configuration:")
print("-" * 40)
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:20s}: {v}")
print("-" * 40)
print(f"\nEstimated training time on P100:")
train_imgs = count_images('train')
estimated_hrs = (train_imgs * 100) / (32 * 60 * 60)  # rough estimate
print(f"  {train_imgs} train images × 100 epochs ≈ {estimated_hrs:.1f} hours")

---
## CELL 12 — START TRAINING 🚀

This is the main training cell. It will take 3–6 hours on Kaggle P100.

In [ ]:
from ultralytics import YOLO
import time

print("="*60)
print("🚀 STARTING YOLOV8 TRAINING")
print("="*60)
print(f"Model:    {TRAIN_CONFIG['model']}")
print(f"Classes:  {NUM_CLASSES}")
print(f"Epochs:   {TRAIN_CONFIG['epochs']}")
print(f"Batch:    {TRAIN_CONFIG['batch']}")
print(f"Images:   {count_images('train')} train / {count_images('val')} val")
print("="*60)
print()

start_time = time.time()

# Load base model (downloads pretrained weights from Ultralytics)
model = YOLO(TRAIN_CONFIG['model'])

# Train
results = model.train(
    data=TRAIN_CONFIG['data'],
    epochs=TRAIN_CONFIG['epochs'],
    patience=TRAIN_CONFIG['patience'],
    imgsz=TRAIN_CONFIG['imgsz'],
    batch=TRAIN_CONFIG['batch'],
    optimizer=TRAIN_CONFIG['optimizer'],
    lr0=TRAIN_CONFIG['lr0'],
    lrf=TRAIN_CONFIG['lrf'],
    momentum=TRAIN_CONFIG['momentum'],
    weight_decay=TRAIN_CONFIG['weight_decay'],
    warmup_epochs=TRAIN_CONFIG['warmup_epochs'],
    warmup_momentum=TRAIN_CONFIG['warmup_momentum'],
    mosaic=TRAIN_CONFIG['mosaic'],
    mixup=TRAIN_CONFIG['mixup'],
    copy_paste=TRAIN_CONFIG['copy_paste'],
    degrees=TRAIN_CONFIG['degrees'],
    translate=TRAIN_CONFIG['translate'],
    scale=TRAIN_CONFIG['scale'],
    fliplr=TRAIN_CONFIG['fliplr'],
    flipud=TRAIN_CONFIG['flipud'],
    hsv_h=TRAIN_CONFIG['hsv_h'],
    hsv_s=TRAIN_CONFIG['hsv_s'],
    hsv_v=TRAIN_CONFIG['hsv_v'],
    device=TRAIN_CONFIG['device'],
    workers=TRAIN_CONFIG['workers'],
    amp=TRAIN_CONFIG['amp'],
    save=TRAIN_CONFIG['save'],
    save_period=TRAIN_CONFIG['save_period'],
    val=TRAIN_CONFIG['val'],
    plots=TRAIN_CONFIG['plots'],
    conf=TRAIN_CONFIG['conf'],
    iou=TRAIN_CONFIG['iou'],
    max_det=TRAIN_CONFIG['max_det'],
    project=TRAIN_CONFIG['project'],
    name=TRAIN_CONFIG['name'],
    exist_ok=True,
    verbose=True,
)

elapsed = time.time() - start_time
print(f"\n✅ Training complete in {elapsed/3600:.2f} hours")

---
## CELL 13 — Evaluate the trained model

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Path to best weights
BEST_WEIGHTS = RUNS_DIR / 'nutrivision_v1' / 'weights' / 'best.pt'
print(f"Best weights: {BEST_WEIGHTS}")
print(f"Exists: {BEST_WEIGHTS.exists()}")

# Load best model
eval_model = YOLO(str(BEST_WEIGHTS))

# Run validation on val split
print("\nRunning validation...")
val_results = eval_model.val(
    data=str(YAML_PATH),
    split='val',
    conf=0.35,
    iou=0.5,
    verbose=True,
)

# Print per-class metrics
print("\n" + "="*60)
print("PER-CLASS METRICS (Validation Set):")
print("="*60)
print(f"{'Class':>20} {'Precision':>12} {'Recall':>10} {'mAP50':>10}")
print("-"*55)

try:
    # Access per-class results
    maps = val_results.box.maps  # per-class mAP50
    ap50 = val_results.box.ap50  # per-class AP50
    
    for i, name in enumerate(CLASS_NAMES):
        map_val = maps[i] if i < len(maps) else 0
        print(f"{name:>20} {'—':>12} {'—':>10} {map_val:>10.4f}")
    
    print("-"*55)
    print(f"{'OVERALL mAP50':>20}: {val_results.box.map50:.4f}")
    print(f"{'OVERALL mAP50-95':>20}: {val_results.box.map:.4f}")
except Exception as e:
    print(f"Could not print per-class metrics: {e}")
    print(f"Overall mAP50: {val_results.box.map50:.4f}")
    print(f"Overall mAP50-95: {val_results.box.map:.4f}")

In [ ]:
# ━━━ Plot training curves ━━━
results_dir = RUNS_DIR / 'nutrivision_v1'

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('NutriVision YOLOv8 Training Results', fontsize=16)

plots = [
    ('results.png', 'Training Results'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('PR_curve.png', 'Precision-Recall Curve'),
]

for i, (fname, title) in enumerate(plots):
    fpath = results_dir / fname
    if fpath.exists():
        img = mpimg.imread(str(fpath))
        ax = axes[i // 3][i % 3]
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'training_summary.png'), dpi=150)
plt.show()
print("✅ Training curves saved")

---
## CELL 14 — Test on sample images (visual verification)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

# Colors per class (BGR for OpenCV)
COLORS = [
    (52, 183, 136),   # rice — green
    (231, 111, 81),   # sambar — orange
    (69, 123, 157),   # aviyal — blue
    (168, 218, 220),  # idli — light blue
    (241, 250, 238),  # coconut_chutney — cream
    (163, 196, 145),  # green_chutney — pale green
    (229, 56, 59),    # chicken_curry — red
    (255, 190, 11),   # biryani — yellow
    (251, 133, 0),    # chapathi — amber
    (131, 56, 236),   # dal_curry — purple
    (58, 134, 255),   # kadala_curry — blue
    (255, 0, 110),    # appam — pink
    (6, 214, 160),    # idiyappam — teal
    (239, 71, 111),   # porotta — coral
    (17, 138, 178),   # beef_curry — dark blue
    (6, 147, 227),    # fish_curry
    (238, 155, 0),    # egg_curry
    (52, 211, 153),   # dosa
    (239, 68, 68),    # vada
    (139, 92, 246),   # puttu
]

def draw_detections(img_path, detections, class_names, colors):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for det in detections:
        cls_id = int(det['cls'])
        conf = float(det['conf'])
        x1, y1, x2, y2 = [int(v) for v in det['xyxy']]
        
        color = colors[cls_id % len(colors)]
        label = f"{class_names[cls_id]} {conf:.0%}"
        
        # Draw box
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        
        # Draw label background
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(img, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(img, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)
    
    return img


# Pick 6 random val images and run inference
val_img_dir = COMBINED_DIR / 'images' / 'val'
val_images = list(val_img_dir.glob('*.jpg')) + list(val_img_dir.glob('*.png'))
sample_images = random.sample(val_images, min(6, len(val_images)))

eval_model = YOLO(str(BEST_WEIGHTS))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('NutriVision — Sample Detections on Validation Images', fontsize=14)

for idx, img_path in enumerate(sample_images):
    results = eval_model(str(img_path), conf=0.30, verbose=False)[0]
    
    detections = []
    for box in results.boxes:
        detections.append({
            'cls':  box.cls[0].item(),
            'conf': box.conf[0].item(),
            'xyxy': box.xyxy[0].tolist(),
        })
    
    annotated = draw_detections(img_path, detections, CLASS_NAMES, COLORS)
    
    ax = axes[idx // 3][idx % 3]
    ax.imshow(annotated)
    ax.set_title(f"{img_path.name[:30]}... ({len(detections)} detections)")
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'sample_detections.png'), dpi=150)
plt.show()
print("✅ Sample detections saved")

---
## CELL 15 — Export model for NutriVision backend

In [ ]:
import shutil
from pathlib import Path

OUTPUT_DIR = BASE_DIR / 'exported_models'
OUTPUT_DIR.mkdir(exist_ok=True)

eval_model = YOLO(str(BEST_WEIGHTS))

# ━━━ Export 1: PyTorch (.pt) — for Flask backend ━━━
pt_dest = OUTPUT_DIR / 'nutrivision_yolov8n_food.pt'
shutil.copy2(str(BEST_WEIGHTS), str(pt_dest))
print(f"✅ PyTorch model (.pt): {pt_dest}")
print(f"   Size: {pt_dest.stat().st_size / 1e6:.1f} MB")

# ━━━ Export 2: ONNX — for cross-platform deployment ━━━
print("\nExporting to ONNX...")
onnx_path = eval_model.export(
    format='onnx',
    imgsz=640,
    opset=17,
    simplify=True,
    dynamic=False,
)
if onnx_path:
    onnx_dest = OUTPUT_DIR / 'nutrivision_yolov8n_food.onnx'
    shutil.copy2(str(onnx_path), str(onnx_dest))
    print(f"✅ ONNX model: {onnx_dest}")
    print(f"   Size: {onnx_dest.stat().st_size / 1e6:.1f} MB")

# ━━━ Export 3: TorchScript — fastest server inference ━━━
print("\nExporting to TorchScript...")
ts_path = eval_model.export(
    format='torchscript',
    imgsz=640,
    optimize=True,
)
if ts_path:
    ts_dest = OUTPUT_DIR / 'nutrivision_yolov8n_food.torchscript'
    shutil.copy2(str(ts_path), str(ts_dest))
    print(f"✅ TorchScript model: {ts_dest}")

# ━━━ Save class names file ━━━
labels_path = OUTPUT_DIR / 'nutrivision_classes.txt'
with open(labels_path, 'w') as f:
    for name in CLASS_NAMES:
        f.write(name + '\n')
print(f"\n✅ Class labels: {labels_path}")

# ━━━ Save model info JSON ━━━
import json
model_info = {
    'model_name': 'nutrivision_yolov8n_food',
    'architecture': 'YOLOv8n',
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'input_size': 640,
    'conf_threshold': 0.35,
    'iou_threshold': 0.45,
    'description': 'Kerala / South Indian food detection for NutriVision app',
    'trained_on': 'Roboflow South Indian + Credo Health + Indian Food Detection datasets',
    'mAP50': float(val_results.box.map50),
    'mAP50_95': float(val_results.box.map),
}
with open(OUTPUT_DIR / 'model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"\n✅ Model info saved")

print("\n" + "="*50)
print("📦 ALL EXPORTS COMPLETE")
print("="*50)
print(f"\nOutput directory: {OUTPUT_DIR}")
for f in OUTPUT_DIR.iterdir():
    size = f.stat().st_size / 1e6
    print(f"  {f.name:45s} {size:6.1f} MB")

print("\n📋 FOR NUTRIVISION BACKEND:")
print(f"   Copy nutrivision_yolov8n_food.pt → backend/models/yolov8_indian_food_seg.pt")
print(f"   Copy nutrivision_classes.txt     → backend/models/labels.txt")

---
## CELL 16 — Quick inference test on a single image

Use this to test the model on your own food photos.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ━━━ TEST ON A SINGLE IMAGE ━━━
# Replace with any image path you want to test
TEST_IMAGE = '/path/to/your/test/image.jpg'   # <-- REPLACE

model = YOLO(str(BEST_WEIGHTS))

# Run inference
results = model(
    TEST_IMAGE,
    conf=0.30,
    iou=0.45,
    max_det=20,
    verbose=False,
)[0]

# Print detections
print(f"Detected {len(results.boxes)} food items:\n")
print(f"{'#':>3} {'Food':>20} {'Confidence':>12} {'Bounding Box (x1,y1,x2,y2)'}")
print("-" * 65)
for i, box in enumerate(results.boxes):
    cls_id = int(box.cls[0])
    conf   = float(box.conf[0])
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    food   = CLASS_NAMES[cls_id]
    print(f"{i+1:>3} {food:>20} {conf:>11.1%}  ({x1},{y1})-({x2},{y2})")

# Show annotated image
annotated_img = results.plot()
plt.figure(figsize=(12, 8))
plt.imshow(annotated_img[:, :, ::-1])  # BGR→RGB
plt.title('NutriVision Food Detection Result')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## CELL 17 — Save all outputs and zip for download

In [ ]:
import shutil

# Create final zip for easy download from Kaggle
print("Creating downloadable zip...")

zip_path = '/kaggle/working/nutrivision_model_export'
shutil.make_archive(
    zip_path,
    'zip',
    str(OUTPUT_DIR),
)

zip_size = Path(zip_path + '.zip').stat().st_size / 1e6
print(f"\n✅ Zip created: {zip_path}.zip ({zip_size:.1f} MB)")
print("\n📥 Download this file from Kaggle's Output tab")
print("   Then extract and copy:")  
print("   nutrivision_yolov8n_food.pt → backend/models/yolov8_indian_food_seg.pt")
print("   nutrivision_classes.txt     → backend/models/labels.txt")
print("   model_info.json             → backend/models/model_info.json")
print()
print("="*60)
print("TRAINING COMPLETE! 🎉")
print(f"Final mAP50:     {val_results.box.map50:.4f}")
print(f"Final mAP50-95:  {val_results.box.map:.4f}")
print("="*60)